### 1. Import libraries


In [1]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score,
                             precision_score,
                             recall_score,
                             f1_score)

### 2. Load data


In [2]:
carpeta_de_datos = "datasets/"
nombre_del_archivo = "users_behavior.csv"
data = pd.read_csv(os.path.join(carpeta_de_datos, nombre_del_archivo))

### 3. Explore data

In [3]:
data.head()

,calls,minutes,messages,mb_used,is_ultra
0,40.0,311.90,83.0,19915.42,0
1,85.0,516.75,56.0,22696.96,0
2,77.0,467.66,86.0,21060.45,0
3,106.0,745.53,81.0,8437.39,1
4,66.0,418.74,1.0,14502.75,0


In [4]:
data.info()

<class 'pandas.DataFrame'>
RangeIndex: 3214 entries, 0 to 3213
Data columns (total 5 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   calls     3214 non-null   float64
 1   minutes   3214 non-null   float64
 2   messages  3214 non-null   float64
 3   mb_used   3214 non-null   float64
 4   is_ultra  3214 non-null   int64  
dtypes: float64(4), int64(1)
memory usage: 125.7 KB


Al ver los datos, entiende rápidamente que este es un problema de clasificación. Por lo tanto se propone:

Objetivo general:

- Predecir si un usuario es un usuario "SMART" o "ULTRA"; en función de la cantidad de llamadas, minutos usados, mensajes enviados y mb de navegación utilizados.

Objetivos específicos:

- Evaluar el balance de los datos (sesgos)
- Realizar una prueba de cordura para cada modelo.
- Entrenar un modelo de clasificación logística (Logistic Regression).
- Entrenar un modelo de árbol de decisión (Decision Tree).
- Entrenar un modelo de bosque aleatorio (Random Forest).
- Calcular el porcentaje de aciertos y la precisión.
- Entrenar cada modelo con diferentes hiperparámetros y comparar los resultados.
    

### 4. Análisis Machine Learning Supervisado

In [5]:
data.head()

,calls,minutes,messages,mb_used,is_ultra
0,40.0,311.90,83.0,19915.42,0
1,85.0,516.75,56.0,22696.96,0
2,77.0,467.66,86.0,21060.45,0
3,106.0,745.53,81.0,8437.39,1
4,66.0,418.74,1.0,14502.75,0


Se observan que los datos tienen diferentes órdenes de magnitud:

decenas en calls y mensajes

centenas en minutes

miles en megas usados

Esto afectará el rendimiento del modelo Logistico, se recomienda estandarizar los datos; para eso se usará StandardScaler

In [6]:
# Evaluar los datos
cantidad_ultra = data[data["is_ultra"]== 1]["is_ultra"].count()
cantidad_smart = data[data["is_ultra"]== 0]["is_ultra"].count()
print(f"Cantidad de usuarios Ultra: {cantidad_ultra}")
print(f"Cantidad de usuarios Smart: {cantidad_smart}")


Cantidad de usuarios Ultra: 985
Cantidad de usuarios Smart: 2229


Se parecia una relación 30/70 entre los usuarios Ultra y Smart, lo que podría afectar el rendimiento del modelo Logistico. Se utilizará class_weight="balanced" para mitigar esa relación.


### 5. Entrenamiento de modelos

#### 5.1. Preparar los datos en datos de entrenamiento, datos de validación y datos de prueba

In [7]:

features = data.drop("is_ultra", axis=1)
target = data["is_ultra"]
x_train, x_temp, y_train, y_temp = train_test_split(features, target, test_size=0.4, random_state=42) # 60% para entrenamiento y 40% para validación y prueba
x_val, x_test, y_val, y_test = train_test_split(x_temp, y_temp, test_size=0.5, random_state=42) # 50% y 50% para validación y prueba respectivamente
# Nota: La estandarización se realiza después de dividir los datos para evitar la fuga de datos (data leakage)
scaler = StandardScaler()
scaler.fit(x_train)
x_train_scaled = scaler.transform(x_train)
x_test_scaled = scaler.transform(x_test)

#### 5.2. Modelo Logistico

In [8]:
# Modelo Logistico
modelo_logistico = LogisticRegression(class_weight='balanced', random_state=42)
modelo_logistico.fit(x_train_scaled, y_train)

y_predict_train = modelo_logistico.predict(x_train_scaled) # Predicciones en el conjunto de entrenamiento
y_predict_test = modelo_logistico.predict(x_test_scaled) # Predicciones en el conjunto de prueba

score_train = accuracy_score(y_train, y_predict_train)
score_test = accuracy_score(y_test, y_predict_test)

precision_train = precision_score(y_train, y_predict_train)
precision_test = precision_score(y_test, y_predict_test)

f1_train = f1_score(y_train, y_predict_train)
f1_test = f1_score(y_test, y_predict_test)

result = {
    "logistic_model": {
        "score_train": score_train,
        "score_test": score_test,
        "precision_train": precision_train,
        "precision_test": precision_test,
        "f1_train": f1_train,
        "f1_test": f1_test
    }
}

#### 5.3. Modelo Decision Tree

##### 5.3.1.Validar el modelo (buscar hyperparámetros)

In [9]:
# NOTA: Decision Tree no necesita estandarización, por lo que se utiliza x_train y x_test sin escalar
best_score = 0
best_depth = 1
best_f1 = 0

for depth in range(1,10):
    modelo_decision_tree = DecisionTreeClassifier(class_weight='balanced', 
                                                  random_state=42, 
                                                  max_depth=depth)
    
    modelo_decision_tree.fit(x_train, y_train)
    
    y_predict_train = modelo_decision_tree.predict(x_train)
    y_predict_val = modelo_decision_tree.predict(x_val)
    
    score_train = accuracy_score(y_train, y_predict_train)
    score_val = accuracy_score(y_val, y_predict_val)

    f1_train = f1_score(y_train, y_predict_train)
    f1_val = f1_score(y_val, y_predict_val)
        
    if abs(f1_train - f1_val) < 0.1 and abs(score_train - score_val) < 0.1 and score_val > best_score:
        best_score = score_val
        best_f1 = f1_val
        best_depth = depth

result["decision_tree_model"] = {
    "score_val": best_score,
    "f1_val": best_f1,
    "best_depth": best_depth
}

##### 5.3.2. Entrenar el modelo con los hyperparámetros elegidos.

In [10]:
depth = result["decision_tree_model"]["best_depth"]

modelo_decision_tree = DecisionTreeClassifier(class_weight='balanced', 
                                                random_state=42, 
                                                max_depth=depth)

modelo_decision_tree.fit(x_train, y_train)

y_predict_train = modelo_decision_tree.predict(x_train)
y_predict_test = modelo_decision_tree.predict(x_test)

score_train = accuracy_score(y_train, y_predict_train)
score_test = accuracy_score(y_test, y_predict_test)

f1_train = f1_score(y_train, y_predict_train)
f1_test = f1_score(y_test, y_predict_test)


result["decision_tree_model"]["score_train"] = score_train
result["decision_tree_model"]["score_test"] = score_test
result["decision_tree_model"]["f1_train"] = f1_train
result["decision_tree_model"]["f1_test"] = f1_test

#### 5.4. Modelo Random Forest

##### 5.4.1. Validar el modelo (buscar hyperparámetros)

In [11]:
# Modelo Random Forest
# Nota: Random Forest no necesita estandarización, por lo que se utiliza x_train y x_test sin escalar
best_score = 0
best_f1_score = 0
best_est = 0
best_depth = 0
best_leaf = 0

for est in [200, 250, 300]:
    for depth in range(3,15, 1):
        for leaf in range(3, 7,1):
            modelo_random_forest = RandomForestClassifier(class_weight='balanced', 
                                                          random_state=42, 
                                                          n_estimators=est,
                                                          max_depth=depth,
                                                          min_samples_split=5,
                                                          min_samples_leaf=leaf)
            modelo_random_forest.fit(x_train, y_train)
            
            y_predict_train = modelo_random_forest.predict(x_train)
            y_predict_val = modelo_random_forest.predict(x_val)
            
            score_train = accuracy_score(y_train, y_predict_train)
            score_val = accuracy_score(y_val, y_predict_val)
            
            f1_train = f1_score(y_train, y_predict_train)
            f1_val = f1_score(y_val, y_predict_val)
            
            if abs(f1_train - f1_val) < 0.1 and abs(score_train - score_val) < 0.1 and score_val > best_score:
                best_score = score_val
                best_f1_score = f1_val
                best_est = est
                best_depth = depth
                best_leaf = leaf
            
result["random_forest_model"] = {
    "score_val": best_score, 
    "f1_val": best_f1_score,
    "best_estimator": best_est,
    "best_depth": best_depth,
    "best_leaf": best_leaf
}

##### 5.4.2. Entrenar el modelo con los hyperparámetros elegidos

In [12]:
leaf = result["random_forest_model"]["best_leaf"]
depth = result["random_forest_model"]["best_depth"]
est = result["random_forest_model"]["best_estimator"]

modelo_random_forest = RandomForestClassifier(class_weight='balanced', 
                                                random_state=42, 
                                                n_estimators=est,
                                                max_depth=depth,
                                                min_samples_split=5,
                                                min_samples_leaf=leaf)

modelo_random_forest.fit(x_train, y_train)

y_predict_train = modelo_random_forest.predict(x_train)
y_predict_test = modelo_random_forest.predict(x_test)

score_train = accuracy_score(y_train, y_predict_train)
score_test = accuracy_score(y_test, y_predict_test)

f1_train = f1_score(y_train, y_predict_train)
f1_test = f1_score(y_test, y_predict_test)
            
result["random_forest_model"]["score_train"] = score_train
result["random_forest_model"]["score_test"] = score_test
result["random_forest_model"]["f1_train"] = f1_train
result["random_forest_model"]["f1_test"] = f1_test

### 6. Mostrar resultados

In [13]:
for key, value in result.items():
    print(f"*****{key.replace('_', ' ').title()}*****")
    for metric_key, metric_value in value.items():
        print(f"{metric_key.replace('_', ' ').title()}: {metric_value:.2f}")
    print("\n")

*****Logistic Model*****
Score Train: 0.64
Score Test: 0.63
Precision Train: 0.44
Precision Test: 0.42
F1 Train: 0.51
F1 Test: 0.49


*****Decision Tree Model*****
Score Val: 0.79
F1 Val: 0.55
Best Depth: 2.00
Score Train: 0.78
Score Test: 0.79
F1 Train: 0.54
F1 Test: 0.58


*****Random Forest Model*****
Score Val: 0.81
F1 Val: 0.66
Best Estimator: 300.00
Best Depth: 6.00
Best Leaf: 4.00
Score Train: 0.83
Score Test: 0.81
F1 Train: 0.70
F1 Test: 0.65




Se observa que el mejor ajuste se obtiene con el modelo Random Forest; con un ajuste de 0.81 en los datos de prueba; y un F1 de 0.65 que indica una predicción buena del plan que usan los clientes en los datos de prueba.

### 7. Prueba de cordura

In [14]:
# 7.1. Hacer dummy baseline: Consiste en comparar el modelo elegido (RandomForest) contra un modelo que no aprende nada (el dummy)
from sklearn.dummy import DummyClassifier

dummy = DummyClassifier(strategy="most_frequent", random_state=42)
dummy.fit(x_train, y_train)
y_predict_dummy = dummy.predict(x_test)
score_dummy = accuracy_score(y_test, y_predict_dummy)
f1_dummy = f1_score(y_test, y_predict_dummy)
print(f"*****Dummy*****")
print(f"Score: {score_dummy:.2f}")
print(f"F1 Score: {f1_dummy:.2f}")

*****Dummy*****
Score: 0.70
F1 Score: 0.00


El dummy detecta que tan dominante es la clase mayoritaria (los usuarios del Plan Smart 2229); y el F1 indica que el modelo no acertó ningún valor de la clase minoritaria (los usuarios del plan Ultra 985).

In [15]:
# 7.2. Revolver los datos (Shuffling): Consiste en ordenar los datos de forma diferente y aleatoria; para comprobar si se obtiene el mismo rendimiento del modelo.
from sklearn.utils import shuffle

# Obtengo los hiperparámetros seleccionados del modelo Random Forest
leaf = result["random_forest_model"]["best_leaf"]
depth = result["random_forest_model"]["best_depth"]
est = result["random_forest_model"]["best_estimator"]

# Usando el train_test_split anterior, ahora solo mezclo el target para intentar hacer fallar al modelo
y_train_shuffled = shuffle(y_train, random_state=42)

# Entreno el modelo Random Forest con los datos mezclados y los hiperparámetros seleccionados
modelo_random_forest = RandomForestClassifier(class_weight='balanced',
                                              random_state=42,
                                              n_estimators=est,
                                              max_depth=depth,
                                              min_samples_split=5,
                                              min_samples_leaf=leaf)

modelo_random_forest.fit(x_train, y_train_shuffled)
y_predict_train_shuffled = modelo_random_forest.predict(x_train)
y_predict_test_shuffled = modelo_random_forest.predict(x_test)

score_train_shuffled = accuracy_score(y_train, y_predict_train_shuffled)
score_test_shuffled = accuracy_score(y_test, y_predict_test_shuffled)

f1_train_shuffled = f1_score(y_train, y_predict_train_shuffled)
f1_test_shuffled = f1_score(y_test, y_predict_test_shuffled)

print(f"*****Random Forest con datos mezclados (Shuffled)*****")
print(f"Score Train: {score_train_shuffled:.2f}")
print(f"F1 Score Train: {f1_train_shuffled:.2f}")
print(f"Score Test: {score_test_shuffled:.2f}")
print(f"F1 Score Test: {f1_test_shuffled:.2f}")

*****Random Forest con datos mezclados (Shuffled)*****
Score Train: 0.59
F1 Score Train: 0.35
Score Test: 0.60
F1 Score Test: 0.36


Esto es un indicador de que se deben verificar con detalle los datos; se espera que F1 fuese cercano a cero pero un valor de 0.36 es muy elevado; además este valor es similar con los datos de entrenamiento y con los de prueba, lo que indica que no es un error o producto del azar. Sería conveniente determinar correlaciones entre las variables "independientes" para eliminar ruido y determinar la mejor forma de clasificar a los usuarios por sus hábitos de consumo.

También vale la pena mencionar que anteriormente, se había detectado que existían varios usuarios de plan SMART que consumían más mb de navegación, cantidad de llamadas y minutos en llamadas que el promedio. También pasaba lo contrario, usuarios con plan ULTRA que no usaban a su máxima capacidad su plan. Estos outliers podrían estar afectando las predicciones

In [ ]:
# 7.3. Crear un overfitting intencional

# Entreno el modelo Random Forest con los datos mezclados y los hiperparámetros seleccionados
modelo_random_forest = RandomForestClassifier(random_state=42,
                                              n_estimators=500,
                                              max_depth=None,
                                              min_samples_split=2,
                                              min_samples_leaf=1,
                                              bootstrap=False)

# modelo_random_forest.fit(x_train[0:101], y_train[0:101])
# modelo_random_forest.fit(x_train[100:201], y_train[100:201])
modelo_random_forest.fit(x_train[500:501], y_train[500:501]) # Fuí ajustando de 0:101 a 100:201 y 500:501 porque los resultados de las pruebas eran superiores a 0.7
y_predict_train = modelo_random_forest.predict(x_train)
y_predict_test = modelo_random_forest.predict(x_test)

score_train = accuracy_score(y_train, y_predict_train)
score_test = accuracy_score(y_test, y_predict_test)

f1_train = f1_score(y_train, y_predict_train)
f1_test = f1_score(y_test, y_predict_test)

print(f"*****Random Forest con overfitting intencional*****")
print(f"Score Train: {score_train:.2f}")
print(f"F1 Score Train: {f1_train:.2f}")
print(f"Score Test: {score_test:.2f}")
print(f"F1 Score Test: {f1_test:.2f}")

*****Random Forest con overfitting intencional*****
Score Train: 0.31
F1 Score Train: 0.47
Score Test: 0.30
F1 Score Test: 0.47


In [ ]:
# Para salir de dudas sobre los resultados elevados utilicé otro aproach sin modificar el modelo
y_train_random = np.random.permutation(y_train)
modelo_random_forest.fit(x_train, y_train_random)
y_pred_train = modelo_random_forest.predict(x_train)
y_pred_test = modelo_random_forest.predict(x_test)
print("Train:", accuracy_score(y_train_random, y_pred_train))
print("Test:", accuracy_score(y_test, y_pred_test))

Train: 1.0
Test: 0.6174183514774495


Esto indica que el modelo está correcto y que si está representando bien a los datos de entrenamiento; sin embargo; la proporción no aporta suficiente información para predecir correctamente. Esto puede deberse en parte a la proporción de los datos 70/30 y a los outliers en los datos.

### 8. Conclusiones

1. Decision Tree fue un modelo que ofreció un buen resultado (accuracy = 0.79, F1=0.62) a un MUY BAJO costo computacional, la operación tardó 0.1 segundos en realizarse. Además, este modelo mostró un ajuste ideal, sin overfitting ni underfitting. Vale la pena mencionar que la prueba F1 tiene un resultado moderado, es decir que el modelo no siempre acierta el tipo de usuario.

2. El modelo Random Forest, ofreció los mejores resultados con los datos de entrenamiento (accuracy = 0.86 y F1 = 0.75); sin embargo, al utilizar los datos de prueba los resultados fueron muy similares a los obtenidos con el Decision Tree (accuracy = 0.8, F1 = 0.65); sin embargo, el costo computacional fue casi 230 veces mayor que el costo computacional del modelo Decision Tree.

3. El peor modelo fue el modelo Logistico con un accuracy de 0.64 y un F1 de 0.49; indicando que el modelo no se ajustó bien a los datos y las predicciones de los planes de los usuarios fue débil.

4. Con respecto al ajuste de los hiperparámetros de los modelos; el modelo Decision Tree indicó que con una profundidad de 2 se obtenía el mejor resultado. Mientras que en el modelo de Random Forest se recorrieron diferentes profundidades, estimadores y número mínimo de hojas; siendo los óptimos 6, 300 y 4 respectivamente.

5. Las comparaciones de los puntajes de las predicciones de los datos de entrenamiento vs los puntajes de las predcciones de los datos de prueba mostraron que no había underfitting ni overfitting en los modelos.

6. La prueba de cordura sólo se le realizó al modelo de Random Forest y determino que:
- El dummy detecta que tan dominante es la clase mayoritaria (los usuarios del Plan Smart 2229); y el F1 indica que el modelo no acertó ningún valor de la clase minoritaria (los usuarios del plan Ultra 985).
- La prueba de shuffle mostró que la métrica F1 fue de 0.36 y se esperaba un valor cercano a 0; esto indica que los datos pueden tener una correlación elevada entre ellos, y puede reafirmar el problema con la distribución de la muestra.
- La prueba del overfitting intencional descartó la existencia de un error en el entrenamiento del modelo y además dió indicios nuevamente de que el dataset presenta un desproporción en la clase objetivo y también correlaciones entre las features (variables "independientes").

7. El modelo que mejor describió los datos fue el de Random Forest el cual ofreció un 86% de aciertos para el total de casos (usuarios con plan Smart o usuaris con plan Ultra); y con un recall de 0.75 que indica que 7.5 veces de cada 10 se logró detectar correctamente si el usuario era del plan Ultra; aún considerando la desproporción de los datos (70/30 Smart/Ultra)